# Product PoC — synthetic-user simulation (NOT research)

**Strict research/product separation.** This notebook is an *internal product PoC*. Every user and interaction here is **simulated** — it stress-tests the whole product (RWE recommender, Information Health Report, AI Coach, recommendation eval, user metrics) *before real users exist*. **None of it is evidence for the paper** — the research runs on real behaviour in `run_mind_eval.ipynb` (MIND clicks) and `run_politosphere_eval.ipynb` (Reddit behaviour).

**Items are real; users are synthetic.** The catalog is Qbias (~21.7k AllSides-labeled articles → gold lean + real outlets + topics); the agents are simulated with independent traits — political viewpoint, topic interests, openness to opposing views, per-outlet trust, article-quality preference, curiosity/novelty, activity level, reading time, and save/share/ignore propensities. Once the MVP has real traffic, these synthetic interactions are replaced with real behaviour.

In [ ]:
# 1) Code
import os
if not os.path.isdir('/content/random_walks_with_erasure'):
    get_ipython().system('git clone --branch claude/sleepy-gates-oecof1 https://github.com/greenwichg/random_walks_with_erasure.git /content/random_walks_with_erasure')
else:
    get_ipython().system('git -C /content/random_walks_with_erasure pull -q')
os.chdir('/content/random_walks_with_erasure')
get_ipython().system('pip install -e . -q')
print('installed ->', os.getcwd())

In [ ]:
# 2) (optional) REAL catalog: Qbias (gold AllSides lean + real outlets + topics).
#    Skip this cell to use a fully-synthetic catalog instead.
import os
QBIAS = 'allsides_balanced_news_headlines-texts.csv'
if not os.path.exists(QBIAS):
    for br in ('main', 'master'):
        get_ipython().system(f"wget -q -O {QBIAS} 'https://raw.githubusercontent.com/irgroup/Qbias/{br}/{QBIAS}'")
        if os.path.exists(QBIAS) and os.path.getsize(QBIAS) > 500000: break
        elif os.path.exists(QBIAS): os.remove(QBIAS)
print('Qbias:', 'ready' if os.path.exists(QBIAS) else 'not downloaded -> synthetic catalog will be used')

In [ ]:
# 3) SIMULATE synthetic users over the catalog. --qbias uses the real Qbias catalog;
#    omit it for a fully-synthetic one. Every output is stamped SIMULATION.
import os
QBIAS = 'allsides_balanced_news_headlines-texts.csv'
arg = f'--qbias {QBIAS}' if os.path.exists(QBIAS) else ''
get_ipython().system(f'python examples/simulate_users.py {arg} --n-users 3000 --max-items 5000 --seed 0 --out-tag sim')

## The product pipeline on synthetic traffic — all SIMULATION
Each cell below is the *real* product code run on the *synthetic* `sim_users.npz`.

In [ ]:
# 4) Recommendation eval (RQ2 accuracy / RQ3 bridging) -- a SYSTEM stress test on
#    synthetic traffic, NOT an accuracy claim (synthetic clicks recover the generative
#    model by construction). Confirms the recommender pipeline runs + the metrics are sane.
get_ipython().system('python examples/eval_mind.py --npz sim_users.npz --no-bprmf')

In [ ]:
# 5) Information Health Report on synthetic readers -- ALL sections populate now:
#    Source Diversity (real outlets, unlike MIND), Topic/Viewpoint/Echo (gold axis),
#    Open-Mindedness (from the sim's shown-vs-clicked slates in sim_behaviors.tsv), and
#    Reporting Ratio / Emotional Balance / Attention (from the SYNTHETIC sim_register.csv /
#    sim_emotion.csv article attributes). --subject-label keeps the header honest (this is
#    synthetic reading over the Qbias catalog, not "MSN-News").
get_ipython().system('python examples/health_report.py --npz sim_users.npz --sample 3 --require-political '
                     '--behaviors sim_behaviors.tsv --register-csv sim_register.csv '
                     '--emotion-csv sim_emotion.csv --subject-label "(simulated) reading diet" '
                     '--html sim_health.html')
from IPython.display import HTML, display
display(HTML(open('sim_health.html').read()))

In [ ]:
# 6) AI Coach (narrative) on a synthetic reader. Free Gemini -- set GEMINI_API_KEY
#    (Colab Secrets, key icon). Narrates ONLY engine-computed numbers; grounding checks
#    flag any it invents.
import os
get_ipython().system('pip -q install google-genai')
if not os.environ.get('GEMINI_API_KEY'):
    try:
        from google.colab import userdata; os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    except Exception: pass
get_ipython().system('python examples/narrate_report.py --npz sim_users.npz')

In [ ]:
# 7) Closed loop: AdaptiveRWEB driven by the SIMULATED cross-cutting reception
#    (cross_welcomed_frac = save/share vs ignore on opposite-side clicks). Adaptive reach
#    should RISE with measured tolerance while uniform stays flat.
get_ipython().system('python examples/adaptive_satisfaction.py --npz sim_users.npz --probe-csv sim_satisfaction_probe.csv')

In [ ]:
# 8) User metrics: the synthetic population's traits + realised behaviour.
import pandas as pd
df = pd.read_csv('sim_population.csv')
print('SIMULATION -- synthetic users. Population summary:')
print(df.describe().round(3).to_string())
print('\nsample readers:'); print(df.head().to_string())